# Cross-sensitivity study: choosing each sensor's gas parents (Issue #16)

**Goal.** Decide which gases each sensor responds to, so the Phase 2 network's edges come from data instead of guesses.

**Why plain correlation isn't enough.** CO, NOx, NO2 and C6H6 all come from traffic, so they rise and fall together, and every sensor looks correlated with every gas. A gas earns an edge only if it tells us something about the sensor **beyond** what the sensor's nominal gas and the weather already explain. We measure that in two ways:

1. **Partial Spearman ρ** of the sensor with the gas, controlling for the nominal gas, T, AH and hour.
2. **BN check**, in the model's own terms: does adding the gas as a parent improve the sensor CPT's held-out log-likelihood? We compare P(S | nominal, g, W) with P(S | nominal, W) on validation rows.

Everything is computed on the **train** split (correlations) or **train → val** (BN check), for split seeds 0, 1 and 2. Test is never touched.

**Sections**
1. Setup and splits
2. Gas–gas correlation (how much the gases co-move)
3. Sensor–gas correlation
4. Partial correlation
5. Is C6H6 a real gas reading? (proxy check)
6. BN check and forward selection
7. Decision: final `nominal` / `cross` edges
8. Summary for the report

## 1. Setup and splits

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cse250a.modules.config import config
from cse250a.modules.core import TIME_CARDS, Gas, Network, Sensor, Time, Weather
from cse250a.modules.data import Discretizer, blocked_split, load_air_quality
from cse250a.modules.eval import partial_spearman, spearman_matrix
from cse250a.modules.models import DiscreteBN

STUDY   = "cross_sensitivity"
RESULTS = config.paths.results_dir / "studies" / STUDY   # CSVs, next to the Makefile's studies/cross_sensitivity.html
FIGS    = config.paths.figures_dir / "studies" / STUDY
for d in (RESULTS, FIGS):
    d.mkdir(parents=True, exist_ok=True)

SEEDS  = (0, 1, 2)

In [ ]:
plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)


def short(name: str) -> str:
    """'PT08.S4(NO2)' -> 'S4 (NO2)', 'NOx(GT)' -> 'NOx'."""
    if name.startswith("PT08."):
        return name[5:7] + " (" + name.split("(")[1]
    return name.split("(")[0]

**Variables.**
- **Weather `W` = T + AH**, as in the current Phase 2 and 3b plans.
- **Candidate gases:** CO, NOx, NO2, C6H6, plus NMHC in the correlation plots only. NMHC is ~90% missing (only Mar–Apr 2004), so it is too sparse for the BN check and is not a Phase 2 node.
- **Nominal targets** (De Vito et al., 2008): S1 → CO, S2 → NMHC, S3 → NOx, S4 → NO2, S5 → O3. NMHC is dropped and there is no O3 reference, so **S2 and S5 have no usable nominal parent**. Their first gas parent is chosen from the data like any other edge.

Rows with missing hour or weather are dropped, because those are always-observed inputs in the network.

In [ ]:
df = load_air_quality(config.paths.air_quality_csv)
W = (Weather.T, Weather.AH)
before = len(df)
df = df.dropna(subset=[Time.HOUR, *W])
print(f"dropped {before - len(df)} rows with missing hour/weather; {len(df)} left")

SENSORS = tuple(Sensor)
CANDIDATES = (Gas.CO, Gas.NOX, Gas.NO2, Gas.C6H6)  # gases in the Phase 2 network
ALL_GASES = (*CANDIDATES, Gas.NMHC)  # NMHC: correlation plots only
NOMINAL = {
    Sensor.S1: (Gas.CO,),
    Sensor.S2: (),  # nominal NMHC is ~90% missing
    Sensor.S3: (Gas.NOX,),
    Sensor.S4: (Gas.NO2,),
    Sensor.S5: (),  # nominal O3 has no reference analyzer
}

splits = {}
for seed in SEEDS:
    train_raw, val_raw, _ = blocked_split(df, seed=seed)  # test is not used here
    disc = Discretizer(n_bins=config.bins.n_bins).fit(
        train_raw, [*SENSORS, *CANDIDATES, *W]
    )
    splits[seed] = {
        "train_raw": train_raw,
        "train": disc.transform(train_raw),
        "val": disc.transform(val_raw),
        "cards": {**disc.cards, **TIME_CARDS},
    }
pd.DataFrame(
    {s: {"train": len(v["train"]), "val": len(v["val"])} for s, v in splits.items()}
).rename_axis("seed", axis=1)

## 2. Gas–gas correlation

How much do the reference gases co-move? Each cell is Spearman ρ on the seed-0 training hours where both gases are observed, with the number of hours `n` underneath.

In [ ]:
train0 = splits[0]["train_raw"]
rho_gg, n_gg = spearman_matrix(train0, ALL_GASES, ALL_GASES)
labels = [short(g) for g in ALL_GASES]
annot = rho_gg.round(2).astype(str) + "\nn=" + n_gg.astype(str)

fig, ax = plt.subplots(figsize=(6, 4.8))
sns.heatmap(
    rho_gg.astype(float),
    annot=annot.values,
    fmt="",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    xticklabels=labels,
    yticklabels=labels,
    square=True,
    cbar_kws={"label": "Spearman ρ"},
    ax=ax,
)
ax.set_title("Gas–gas Spearman ρ (train, seed 0)")
plt.tight_layout()
plt.savefig(config.paths.figure(FIGS, "gas_gas_spearman.png"), bbox_inches="tight")
plt.show()

**Explanation:** every pair of traffic gases is strongly correlated (ρ ≈ 0.7–0.9). This is why raw sensor–gas correlation can't pick edges on its own: a sensor that only responds to CO will still look correlated with NOx, NO2 and C6H6. NMHC pairs rest on only a few hundred hours, so treat them as low confidence.

## 3. Sensor–gas correlation

In [ ]:
rho_sg, n_sg = spearman_matrix(train0, SENSORS, ALL_GASES)
annot = rho_sg.round(2).astype(str) + "\nn=" + n_sg.astype(str)

fig, ax = plt.subplots(figsize=(7, 4.6))
sns.heatmap(
    rho_sg.astype(float),
    annot=annot.values,
    fmt="",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    xticklabels=labels,
    yticklabels=[short(s) for s in SENSORS],
    cbar_kws={"label": "Spearman ρ"},
    ax=ax,
)
for i, s in enumerate(SENSORS):  # outline each sensor's nominal gas
    for g in NOMINAL[s]:
        ax.add_patch(
            plt.Rectangle((ALL_GASES.index(g), i), 1, 1, fill=False, lw=2.5, ec="black")
        )
ax.set_title("Sensor–gas Spearman ρ (train, seed 0); boxes = nominal gas")
plt.tight_layout()
plt.savefig(config.paths.figure(FIGS, "sensor_gas_spearman.png"), bbox_inches="tight")
plt.show()

**Explanation:** almost every cell is strong, including off-nominal ones. Three things stand out:
- **S4's nominal gas is the weakest link:** ρ(S4, NO2) is about 0.3, while S4 tracks CO and C6H6 far better.
- **S3 is negatively correlated with everything**, because its resistance falls as oxidizing gases rise.
- **S2 vs C6H6 is ρ = 1.00.** Section 5 explains why.

## 4. Partial correlation

Partial Spearman ρ of each sensor with each gas, **controlling for the sensor's nominal gas + T + AH + hour**. Hour enters as 24 dummies, so the shared rush-hour pattern is removed too. For S2 and S5, which have no nominal gas, the controls are weather and hour only. Each cell shows the mean ± std over seeds 0–2.

In [ ]:
def partial_row(train_raw, sensor, gas, base):
    return partial_spearman(
        train_raw, sensor, gas, [*base, *W, Time.HOUR], categorical=[Time.HOUR]
    )


partial = {}
for s in SENSORS:
    for g in ALL_GASES:
        if g in NOMINAL[s]:
            continue
        vals = [partial_row(splits[k]["train_raw"], s, g, NOMINAL[s]) for k in SEEDS]
        partial[(s, g)] = (
            np.mean([v[0] for v in vals]),
            np.std([v[0] for v in vals]),
            vals[0][1],
        )

mean = pd.DataFrame(np.nan, index=SENSORS, columns=ALL_GASES)
text = pd.DataFrame("nominal", index=SENSORS, columns=ALL_GASES)
for (s, g), (m, sd, n) in partial.items():
    mean.loc[s, g] = m
    text.loc[s, g] = f"{m:.2f}\n±{sd:.2f}"

fig, ax = plt.subplots(figsize=(7, 4.6))
sns.heatmap(
    mean.astype(float),
    annot=text.values,
    fmt="",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    xticklabels=labels,
    yticklabels=[short(s) for s in SENSORS],
    cbar_kws={"label": "partial ρ"},
    ax=ax,
)
ax.set_title("Partial Spearman ρ | nominal gas + T + AH + hour (mean ± std, seeds 0–2)")
plt.tight_layout()
plt.savefig(config.paths.figure(FIGS, "partial_spearman.png"), bbox_inches="tight")
plt.show()

**Explanation:**
- Once the nominal gas and weather are held fixed, most of S1's off-nominal correlations shrink a lot (NOx ≈ 0.02).
- S4 keeps a large partial ρ with CO (≈ 0.67) even after controlling for NO2, which is real cross-sensitivity.
- S2 and S5 have no nominal control, so their row stays high for every gas. For those two sensors, the forward selection in section 6 does the real work: it picks the best gas first and then asks what the others add beyond it.
- C6H6 looks strong for every sensor. Section 5 shows why that is misleading.

## 5. Is C6H6 a real gas reading? (proxy check)

The EDA (Issue #4) found that C6H6(GT) is an almost exact function of S2: a cubic fit gives R² ≈ 0.99998, and both are missing on the same rows. If C6H6 is really "S2's reading in disguise", then its correlation with any other sensor should be inherited entirely from S2. To test this, we control for S2 as well, and C6H6's partial ρ should drop to zero.

In [ ]:
rows = {}
for s in SENSORS:
    if s == Sensor.S2:
        continue
    base = [*NOMINAL[s], *W, Time.HOUR]
    without = [
        partial_spearman(
            splits[k]["train_raw"], s, Gas.C6H6, base, categorical=[Time.HOUR]
        )[0]
        for k in SEEDS
    ]
    with_s2 = [
        partial_spearman(
            splits[k]["train_raw"],
            s,
            Gas.C6H6,
            [*base, Sensor.S2],
            categorical=[Time.HOUR],
        )[0]
        for k in SEEDS
    ]
    via_s2 = [
        partial_spearman(
            splits[k]["train_raw"], s, Sensor.S2, base, categorical=[Time.HOUR]
        )[0]
        for k in SEEDS
    ]
    rows[short(s)] = {
        "partial ρ(S, C6H6)": np.mean(without),
        "… also controlling S2": np.mean(with_s2),
        "partial ρ(S, S2 sensor)": np.mean(via_s2),
    }
proxy = pd.DataFrame(rows).T.round(3)
proxy

**Explanation:** C6H6's partial ρ with S1, S3, S4 and S5 is **identical to S2's own**, and it falls to about 0.01 once S2 is controlled for. So C6H6(GT) carries no information that S2 doesn't already have. An edge C6H6 → S4, for example, would really be modeling the correlation between the S2 and S4 sensors, not a sensitivity of S4 to benzene.

**Consequence:** C6H6 is allowed **only as a parent of S2**. That edge is structural rather than a measured cross-sensitivity: it is how the benzene node gets sensor evidence at all, and the report should say so. For every other sensor, C6H6 is excluded from the candidates.

## 6. BN check and forward selection

For a sensor `S` with current gas parents `base` and a candidate `g`, we fit two one-node networks on train and score them on val:

- `P(S | base, W)` versus `P(S | base, g, W)`
- **3 bins, Laplace α = 1** (the project defaults)
- **Same rows for both:** hours where S, base, g, T and AH are all observed in that split

The score is the gain in mean validation log-likelihood per hour (nats). We use **forward selection**: at each step we add the best candidate, but only if it passes the rule below, and then we re-test the rest against the new base. This is the same "beyond what's already explained" idea as the partial correlation, applied step by step.

**Rule for keeping an edge** (all must hold):
1. Δ log-lik ≥ 0.02 nats/hour on average, and positive in every seed (mean − 2·std > 0).
2. |partial ρ| ≥ 0.1, controlling for base + W + hour, with the same sign in all three seeds.
3. At most 2 gas parents beyond the nominal one. Each extra parent multiplies the CPT by 3 (the number of bins).
4. C6H6 is a candidate for S2 only (section 5), and NMHC is never a candidate.

In [ ]:
MIN_GAIN, MIN_PARTIAL, MAX_CROSS = 0.02, 0.10, 2


def val_loglik(sensor, parents, train, val, cards):
    net = Network(structure={sensor: tuple(parents)}, cards=cards).validate(
        train.columns
    )
    bn = DiscreteBN(net, alpha=config.bins.alpha).fit(train)
    return bn.log_likelihood(val) / len(val)


def edge_gain(sensor, base, gas, seed):
    """Val log-lik gain per hour of adding `gas` to P(sensor | base, W), on shared rows."""
    sp = splits[seed]
    cols = [sensor, *base, gas, *W]
    train, val = sp["train"].dropna(subset=cols), sp["val"].dropna(subset=cols)
    before = val_loglik(sensor, (*base, *W), train, val, sp["cards"])
    after = val_loglik(sensor, (*base, gas, *W), train, val, sp["cards"])
    return after - before, len(val)


def candidates_for(sensor):
    return [
        g
        for g in CANDIDATES
        if g not in NOMINAL[sensor] and (g != Gas.C6H6 or sensor == Sensor.S2)
    ]


def select(sensor):
    base, log = list(NOMINAL[sensor]), []
    while True:
        remaining = [g for g in candidates_for(sensor) if g not in base]
        n_cross = len(base) - len(NOMINAL[sensor])
        if not remaining:
            break
        step = []
        for g in remaining:
            gains = [edge_gain(sensor, base, g, k) for k in SEEDS]
            prs = [
                partial_spearman(
                    splits[k]["train_raw"],
                    sensor,
                    g,
                    [*base, *W, Time.HOUR],
                    categorical=[Time.HOUR],
                )[0]
                for k in SEEDS
            ]
            d = np.array([x[0] for x in gains])
            step.append(
                {
                    "sensor": short(sensor),
                    "gas": short(g),
                    "given": ", ".join(short(b) for b in base) or "(none)",
                    "ρ": rho_sg.loc[sensor, g],
                    "partial ρ": np.mean(prs),
                    "partial ρ std": np.std(prs),
                    "Δ loglik": d.mean(),
                    "Δ loglik std": d.std(),
                    "Δ per seed": np.round(d, 3).tolist(),
                    "val n": gains[0][1],
                    "CPT rows added": int(
                        np.prod([3] * (len(base) + 1 + len(W)))
                        - np.prod([3] * (len(base) + len(W)))
                    ),
                    "_g": g,
                    "_ok": d.mean() >= MIN_GAIN
                    and d.mean() - 2 * d.std() > 0
                    and (d > 0).all()
                    and abs(np.mean(prs)) >= MIN_PARTIAL
                    and len({np.sign(p) for p in prs}) == 1,
                }
            )
        step.sort(key=lambda r: -r["Δ loglik"])
        best = step[0]
        for r in step:
            if r is best and r["_ok"] and n_cross < MAX_CROSS:
                r["decision"] = "kept"
            elif r is best and r["_ok"]:
                r["decision"] = "rejected: cap of 2 cross edges"
            elif r is best:
                r["decision"] = "rejected: fails rule"
            else:
                r["decision"] = (
                    "re-tested next step"
                    if best["decision"] == "kept"
                    else "rejected: fails rule"
                )
        log += step
        if best["decision"] != "kept":
            break
        base.append(best["_g"])
    return tuple(base), log


chosen, steps = {}, []
for s in SENSORS:
    chosen[s], log = select(s)
    steps += log
    print(f"{short(s):9s} -> {[short(g) for g in chosen[s]]}")

In [ ]:
# Add the edges excluded up front, so the table lists every candidate
for s in SENSORS:
    if s != Sensor.S2:
        m = partial[(s, Gas.C6H6)]
        steps.append(
            {
                "sensor": short(s),
                "gas": "C6H6",
                "given": ", ".join(short(b) for b in NOMINAL[s]) or "(none)",
                "ρ": rho_sg.loc[s, Gas.C6H6],
                "partial ρ": m[0],
                "partial ρ std": m[1],
                "decision": "rejected: C6H6 only proxies S2 (section 5)",
            }
        )
    m = partial[(s, Gas.NMHC)]
    steps.append(
        {
            "sensor": short(s),
            "gas": "NMHC",
            "given": ", ".join(short(b) for b in NOMINAL[s]) or "(none)",
            "ρ": rho_sg.loc[s, Gas.NMHC],
            "partial ρ": m[0],
            "partial ρ std": m[1],
            "decision": "rejected: NMHC ~90% missing, not a Phase 2 node",
        }
    )

table = pd.DataFrame(steps).drop(columns=["_g", "_ok"])
# a candidate re-tested at a later step is reported once, at its last test
table = table[table["decision"] != "re-tested next step"]
table = table.sort_values(
    ["sensor", "decision"],
    key=lambda c: c.map(lambda v: (v != "kept", v)) if c.name == "decision" else c,
)
table.to_csv(config.paths.figure(FIGS, "candidate_edges.csv"), index=False)
show = table.copy()
for c in ["ρ", "partial ρ", "partial ρ std", "Δ loglik", "Δ loglik std"]:
    show[c] = show[c].astype(float).round(3)
show.astype(object).fillna("").reset_index(drop=True)

In [ ]:
# Δ log-lik for every candidate tested by the BN check (mean ± std over seeds)
bn_rows = table.dropna(subset=["Δ loglik"]).copy()
bn_rows["label"] = (
    bn_rows["sensor"] + " ← " + bn_rows["gas"] + "  | " + bn_rows["given"]
)
bn_rows = bn_rows.iloc[::-1]
colors = ["#2E7D32" if d == "kept" else "#9E9E9E" for d in bn_rows["decision"]]

fig, ax = plt.subplots(figsize=(8, 0.42 * len(bn_rows) + 1))
ax.barh(
    bn_rows["label"],
    bn_rows["Δ loglik"],
    xerr=bn_rows["Δ loglik std"],
    color=colors,
    capsize=3,
)
ax.axvline(0, color="black", lw=0.8)
ax.axvline(
    MIN_GAIN, color="#C62828", ls="--", lw=1, label=f"threshold {MIN_GAIN} nats/hour"
)
ax.set_xscale("symlog", linthresh=0.05)
ax.set_xlabel("Δ val log-likelihood per hour (nats), symlog scale")
ax.set_title("BN check: gain from adding each gas parent (green = kept)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(config.paths.figure(FIGS, "bn_loglik_gain.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Fallback for S2 if the team drops C6H6: best gas parent with no base
fallback = {
    short(g): np.mean([edge_gain(Sensor.S2, (), g, k)[0] for k in SEEDS])
    for g in (Gas.CO, Gas.NOX, Gas.NO2)
}
pd.Series(fallback, name="Δ loglik for S2, no other gas parent").round(3)

**Explanation:**
- **S1 ← CO** alone. NO2 keeps a partial ρ of about 0.24, but at 3 bins it doesn't improve the CPT's held-out fit, so it fails the rule. NOx adds nothing.
- **S2 ← C6H6.** This edge is structural (section 5). Once C6H6 is in, no other gas adds anything. If the team ever drops C6H6, CO is the clear replacement (partial ρ ≈ 0.92, Δ ≈ 0.55 nats/hour).
- **S3 ← NOx** alone. CO and NO2 have partial ρ of about −0.34, but their log-lik gains swing across seeds (the error bar crosses the threshold), so they are not reliable at 3 bins.
- **S4 ← NO2 + CO + NOx.** This is the strongest cross-sensitivity in the data. CO adds about 0.16 nats/hour beyond NO2, and NOx adds a small but very stable gain on top (≈ 0.033 ± 0.003). It confirms the old prior guess S4 ← CO.
- **S5 ← CO + NOx.** S5 has no reference for its nominal gas, so CO comes in first. NOx then adds a consistent gain. **NO2 does not:** after CO and NOx, its gain is negative. This refines Kiernan's EDA proposal of NO2 → S5. His regression didn't include CO or NOx, and NOx already contains NO2 (NOx = NO + NO2), so the NO2 signal in S5 is covered by NOx.

## 7. Decision: final `nominal` / `cross` edges

In [ ]:
nominal = {s: NOMINAL[s] for s in SENSORS if NOMINAL[s]}
cross = {s: tuple(g for g in chosen[s] if g not in NOMINAL[s]) for s in SENSORS}
cross = {s: gs for s, gs in cross.items() if gs}

print("nominal = {")
for s, gs in nominal.items():
    print(f"    Sensor.{s.name}: ({', '.join('Gas.' + g.name for g in gs)},),")
print("}")
print("cross = {")
for s, gs in cross.items():
    print(f"    Sensor.{s.name}: ({', '.join('Gas.' + g.name for g in gs)},),")
print("}")

Copy these two dicts into the Phase 2 notebook. The sensor nodes there become
`{s: (*nominal.get(s, ()), *cross.get(s, ()), *W) for s in Sensor}`, the same pattern the Phase 3b issue (#19) uses.

| Sensor | Nominal (solid) | Cross-sensitivity (dashed) | Gas parents | CPT parent combos (3 bins, T + AH) |
|---|---|---|---|---|
| S1 (CO) | CO | none | 1 | 27 |
| S2 (NMHC) | none (NMHC dropped) | C6H6 (structural) | 1 | 27 |
| S3 (NOx) | NOx | none | 1 | 27 |
| S4 (NO2) | NO2 | CO, NOx | 3 | 243 |
| S5 (O3) | none (no O3 reference) | CO, NOx | 2 | 81 |

**Prior guesses from the old `CROSS` table:**

| Prior guess | Result |
|---|---|
| S2 ← C6H6 | ✅ kept, but as a structural edge |
| S4 ← CO | ✅ confirmed |
| S5 ← NO2 | ❌ replaced by S5 ← CO, NOx |

**Figure change for Phase 2:** remove the dashed CO → S2 edge, and move S5's dashed edge so it comes from CO and NOx. The figure's current NOx → S5 edge stays. Add dashed CO → S4 and NOx → S4 edges.

## 8. Summary for the report

- **Method.** Raw sensor–gas correlation can't select edges, because the traffic gases co-move (ρ ≈ 0.7–0.9). We kept a gas as a sensor parent only if it explained the sensor beyond its nominal gas and the weather. Two tests had to agree: partial Spearman ρ (controlling for nominal gas, T, AH and hour) and a gain in the sensor CPT's held-out log-likelihood. The edges were added by forward selection and had to hold across three split seeds.
- **Main findings.**
  - S1 and S3 are well explained by their nominal gas alone.
  - S4 responds much more to CO than to its nominal NO2, which explains why NO2 is the hardest gas to read from the sensors.
  - S5 behaves like a general traffic sensor (CO + NOx).
- **The benzene caveat.** The C6H6 reference is effectively a transform of S2. Its correlation with every other sensor disappears once S2 is controlled for. C6H6 → S2 is therefore kept only as a structural edge, and benzene "predictions" in later phases are essentially a re-reading of S2. This belongs in Limitations.
- **Edges rejected despite high raw correlation** (good material for Analysis & Interpretation):
  - S1 ← C6H6 (raw ρ ≈ 0.88)
  - S5 ← NO2 (raw ρ ≈ 0.77)
  - S3 ← CO (raw ρ ≈ −0.80)

  Each looks strong on its own, but adds little once the shared traffic signal is accounted for. This is d-separation in practice: two sensors with a common gas parent look correlated, even though neither causes the other.
- **Limits.**
  - Decisions are at 3 bins. With more bins, borderline edges like S1 ← NO2 or S3 ← CO might pass.
  - Partial correlation is rank-linear, while the BN check is not, which is why both are used.
  - NMHC is excluded for lack of data.

*AI use: analysis drafted with help from Claude, reviewed and run by me.*